# StackOverflow Plain Files to Lucene Index

This notebook does two things:
1. Convert StackOverflow plain/Solr files into FlexNeuART JSONL collection layout.
2. Build a Lucene index from the converted collection.

In [16]:
import os
import re
import json
import html
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'

# Root where your plain split folders exist: train/test/dev1/dev2/tran
STACKOVERFLOW_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/output/stackoverflow')

# Split to convert/index in this run
SPLIT_NAME = 'tran'

# created done: train test dev1 dev2 tran

# Collection root used by FlexNeuART scripts
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()
os.makedirs(COLLECT_ROOT, exist_ok=True)
os.environ['COLLECT_ROOT'] = str(COLLECT_ROOT)

# Name of the generated collection
COLLECT_NAME = f'stackoverflow_{SPLIT_NAME}'
COLLECT_DIR = COLLECT_ROOT / COLLECT_NAME
OUT_SPLIT_DIR = COLLECT_DIR / 'input_data' / SPLIT_NAME
OUT_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT    =', REPO_ROOT)
print('SCRIPTS_DIR  =', SCRIPTS_DIR)
print('COLLECT_ROOT =', COLLECT_ROOT)
print('SPLIT_NAME   =', SPLIT_NAME)
print('COLLECT_NAME =', COLLECT_NAME)
print('OUT_SPLIT_DIR=', OUT_SPLIT_DIR)

REPO_ROOT    = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
SCRIPTS_DIR  = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
COLLECT_ROOT = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
SPLIT_NAME   = tran
COLLECT_NAME = stackoverflow_tran
OUT_SPLIT_DIR= /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/input_data/tran


In [13]:
split_dir = STACKOVERFLOW_ROOT / SPLIT_NAME
solr_q = split_dir / 'SolrQuestionFile.txt'
solr_a = split_dir / 'SolrAnswerFile.txt'
qrels_src = split_dir / 'qrels.txt'

for p in [solr_q, solr_a, qrels_src]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

tag_re_cache = {}
def get_tag(line: str, tag: str) -> str:
    if tag not in tag_re_cache:
        tag_re_cache[tag] = re.compile(fr'<{tag}>(.*?)</{tag}>')
    m = tag_re_cache[tag].search(line)
    return html.unescape(m.group(1)) if m else ''

def convert_solr_file(inp: Path, out_jsonl: Path):
    qty = 0
    with inp.open('r', encoding='utf-8', errors='ignore') as fi, out_jsonl.open('w', encoding='utf-8') as fo:
        for line in fi:
            line = line.strip()
            if not line:
                continue

            docno = get_tag(line, 'DOCNO')
            text = get_tag(line, 'Text_bm25')
            text_unlemm = get_tag(line, 'TextUnlemm_bm25')
            bigram = get_tag(line, 'BiGram_bm25')

            if not docno:
                continue

            doc = {
                'DOCNO': docno,
                'text': text,
                'text_unlemm': text_unlemm,
                'text_raw': text_unlemm if text_unlemm else text,
                'bigram': bigram
            }
            fo.write(json.dumps(doc, ensure_ascii=True) + '\n')
            qty += 1
    return qty

q_out = OUT_SPLIT_DIR / 'QuestionFields.jsonl'
a_out = OUT_SPLIT_DIR / 'AnswerFields.jsonl'

q_qty = convert_solr_file(solr_q, q_out)
a_qty = convert_solr_file(solr_a, a_out)
shutil.copy2(qrels_src, OUT_SPLIT_DIR / 'qrels.txt')

print('Wrote queries:', q_qty, '->', q_out)
print('Wrote answers:', a_qty, '->', a_out)
print('Copied qrels ->', OUT_SPLIT_DIR / 'qrels.txt')

Wrote queries: 5772614 -> /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/input_data/tran/QuestionFields.jsonl
Wrote answers: 5772614 -> /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/input_data/tran/AnswerFields.jsonl
Copied qrels -> /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/input_data/tran/qrels.txt


In [14]:
# Build Lucene index for field 'text'
cmd = [
    'bash', './index/create_lucene_index.sh',
    COLLECT_NAME,
    '-input_subdir', 'input_data',
    '-index_field', 'text'
]

print('Running in', SCRIPTS_DIR)
print('Command:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Lucene indexing failed with code {res.returncode}')

Running in /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
Command: bash ./index/create_lucene_index.sh stackoverflow_tran -input_subdir input_data -index_field text
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
Input data directory: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/input_data
Index directory:      /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/lucene_index
Index field name:     text
Exact match param:    
Ignore missing field: 
Checking input sub-directory: tran
Found indexable data file: tran/AnswerFields.jsonl
Found query file: tran/QuestionFields.jsonl
Using the data input file: AnswerFields.jsonl
JAVA_OPTS=-Xms4070758k -Xmx28495306k -server



In [15]:
lucene_dir = COLLECT_DIR / 'lucene_index'
print('Lucene index dir:', lucene_dir)
print('Exists:', lucene_dir.exists())
if lucene_dir.exists():
    children = sorted([p.name for p in lucene_dir.iterdir()])
    print('Files:', children[:20])

Lucene index dir: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/lucene_index
Exists: True
Files: ['_0.cfe', '_0.cfs', '_0.si', 'segments_1', 'write.lock']


## Notes
- To index another split, only change `SPLIT_NAME` in Cell 2 and rerun from Cell 2 onward.
- To index non-lemmatized text, change `-index_field text` to `-index_field text_unlemm` in Cell 4.